In [ ]:
# === PARAMETERS ===
MODEL_TYPE = "COORD" 
LOG_FILE = "test_file.txt"
NAME_ID = "val_dec_max_v1"

# Grid / Agents
WIDTH = 6
HEIGHT = 6
NUM_AGENTS = 2
SPAWN_PROB = 0.05
DESPAWN_PROB = 0.09

# Model Config (High Capacity)
MLP_HIDDEN_DIM = 256
MLP_NUM_LAYERS = 2
CNN_CONV_CHANNELS = [16, 32]
CNN_HEAD_HIDDEN_DIM = 256
CNN_HEAD_NUM_LAYERS = 2
CNN_KERNEL_SIZE = 3

# Value Learning Config
LR = 0.00002
BATCH_SIZE = 512
DISCOUNT = 0.99
TARGET_UPDATE_FREQ = 1000
BUFFER_SIZE = 50000
SEED = 42
MAX_STEPS = 10000000  # Infinite
EVAL_FREQ = 10000 #10000
EVAL_STEPS = 2000

# Epsilon Greedy
EPSILON = 0.1
SAVE_FREQ = 50000

LOAD_MODEL_FOLDER = "None" 
START_STEP = 0

In [ ]:
import os
import torch

# === 1. Force Safe Memory Configuration ===
# This addresses the "misaligned address" on RTX 4090
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

# === 2. Disable Aggressive Optimizations ===
# Stop PyTorch from hunting for "fast" kernels that crash
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.enabled = True 

# === 3. Disable TF32 (The likely culprit) ===
# Ada Lovelace (4090) uses TF32 by default, which has strict alignment rules.
# Disabling it forces FP32, which is safer.
torch.backends.cudnn.allow_tf32 = False
torch.backends.cuda.matmul.allow_tf32 = False

print("NUCLEAR SAFETY FLAGS APPLIED: Benchmark=False, TF32=False, Deterministic=True")

In [ ]:
OUT_FOLDER = f"value_decentralized_{MODEL_TYPE}_lr{LR}_{NAME_ID}"

In [ ]:
from pathlib import Path
import sys
import numpy as np
import torch
import random
import ast
import math
from tqdm import tqdm



sys.path.append("../") 
from config import DEVICE 
from tadd_helpers.env_functions import init_empty_state, spawn_apples, despawn_apples, State
from tadd_helpers.eval_functions import nearest_policy, random_policy
from models.value_mlp import ValueMLPDecentralized
from models.value_cnn_new import ValueCNNDecentralizedStandard, ValueCNNCoordDecentralized
from orchard.environment import MoveAction


if isinstance(CNN_CONV_CHANNELS, str):
    CNN_CONV_CHANNELS = ast.literal_eval(CNN_CONV_CHANNELS)

torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

OUT_FOLDER = Path(OUT_FOLDER)
OUT_FOLDER.mkdir(parents=True, exist_ok=True)
LOG_FILE = OUT_FOLDER / LOG_FILE

In [ ]:
from models.value_cnn_new import ReplayBuffer

# --- INITIALIZATION (N Independent Learners) ---
models = []

for i in range(NUM_AGENTS):
    if MODEL_TYPE == "MLP":
        m = ValueMLPDecentralized(
            height=HEIGHT, width=WIDTH, lr=LR, discount=DISCOUNT,
            hidden_dim=MLP_HIDDEN_DIM, num_layers=MLP_NUM_LAYERS
        )
    elif MODEL_TYPE == "CNN":
        m = ValueCNNDecentralizedStandard(
            height=HEIGHT, width=WIDTH, lr=LR, discount=DISCOUNT,
            hidden_dim=CNN_HEAD_HIDDEN_DIM, num_layers=CNN_HEAD_NUM_LAYERS, 
            conv_channels=CNN_CONV_CHANNELS, kernel_size=CNN_KERNEL_SIZE
        )
    elif MODEL_TYPE == "COORD":
        m = ValueCNNCoordDecentralized(
            height=HEIGHT, width=WIDTH, lr=LR, discount=DISCOUNT,
            hidden_dim=CNN_HEAD_HIDDEN_DIM, num_layers=CNN_HEAD_NUM_LAYERS, 
            conv_channels=CNN_CONV_CHANNELS, kernel_size=CNN_KERNEL_SIZE
        )
    else:
        raise ValueError(f"Unknown MODEL_TYPE: {MODEL_TYPE}")
    # Expand Buffer for each agent
    m.memory = ReplayBuffer(BUFFER_SIZE)
    models.append(m)

print(f"Initialized {NUM_AGENTS} Decentralized {MODEL_TYPE} models. Total Buffer: 1M.")

In [ ]:
# === RESUME LOGIC ===
if LOAD_MODEL_FOLDER != "None" and str(LOAD_MODEL_FOLDER) != "None":
    print(f"RESUMING agents from folder: {LOAD_MODEL_FOLDER} at Step {START_STEP}")
    from pathlib import Path
    
    folder_path = Path(LOAD_MODEL_FOLDER)
    
    for i, m in enumerate(models):
        # Construct path: folder/models/model_agent{i}_step{START_STEP}.pt
        chk_path = folder_path / "models" / f"model_agent{i}_step{START_STEP}.pt"
        
        if not chk_path.exists():
            print(f"WARNING: Could not find {chk_path}")
        else:
            state_dict = torch.load(chk_path, map_location=DEVICE)
            m.load_state_dict(state_dict)
            m.update_target_net()
            print(f"Loaded Agent {i} from {chk_path}")

In [ ]:
print("Checking model weights for NaNs...")
has_nans = False
for i, m in enumerate(models):
    for name, param in m.named_parameters():
        if torch.isnan(param).any():
            print(f"CRITICAL ERROR: Agent {i} has NaNs in layer {name}!")
            has_nans = True

if has_nans:
    print("❌ CONCLUSION: The save file is corrupt. You must load an OLDER checkpoint (e.g. 6,200,000).")
else:
    print("✅ CONCLUSION: Weights are clean. The crash is due to a specific input/seed collision.")

In [ ]:
import psutil
import os
def get_ram_usage():
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / 1024 / 1024  # Returns MB

In [ ]:
def get_best_action_decentralized_lookahead(models_list, state: State, agent_idx: int):
    """
    Decentralized Lookahead:
    Queries each agent's specific model for the Team Value sum.
    """
    agent_pos = state.agent_position(agent_idx)
    candidates = []
    
    for action in MoveAction:
        new_pos = np.clip(agent_pos + action.vector, [0, 0], [HEIGHT-1, WIDTH-1])
        
        s_mid = state.copy()
        s_mid.set_agent_position(agent_idx, new_pos)
        
        # Calculate Team Value (Sum of V_i from Model_i)
        current_team_val = 0.0
        for i in range(NUM_AGENTS):
            pos_i = s_mid.agent_position(i)
            # Query the specific model for agent i
            current_team_val += models_list[i].get_value(s_mid, agent_pos=pos_i)
            
        candidates.append((current_team_val, action))
            
    # --- CRASH PROOFING START ---
    # Filter out NaNs (Not a Number) or Infs (Infinity)
    valid_candidates = []
    for val, act in candidates:
        if not np.isnan(val) and not np.isinf(val):
            valid_candidates.append((val, act))
            
    # If model is broken (all NaNs), just pick random to survive
    if len(valid_candidates) == 0:
        return random.choice(list(MoveAction))
        
    # Standard logic
    max_val = max(c[0] for c in valid_candidates)
    best_actions = [act for (val, act) in valid_candidates if abs(val - max_val) < 1e-5]
    
    if len(best_actions) == 0:
        return random.choice(list(MoveAction))
    # --- CRASH PROOFING END ---
    
    return random.choice(best_actions)

def learned_policy_wrapper(s, idx):
    return get_best_action_decentralized_lookahead(models, s, idx)

# Re-define eval to use the wrapper
def run_evaluation_episode(policy_func, steps=500):
    s = init_empty_state(HEIGHT, WIDTH, NUM_AGENTS)
    total_reward = 0.0
    for _ in range(steps):
        active_idx = s.get_random_agent_id()
        action = policy_func(s, active_idx)
        new_pos = np.clip(s.agent_position(active_idx) + action.vector, [0, 0], [HEIGHT-1, WIDTH-1])
        s.set_agent_position(active_idx, new_pos)
        step_reward = 0.0
        if s.apples[tuple(new_pos)] > 0:
            s.remove_apple_at(new_pos)
            step_reward = 1.0
        total_reward += step_reward
        spawn_apples(s, SPAWN_PROB)
        despawn_apples(s, DESPAWN_PROB)
    return total_reward / steps

In [ ]:
loss_history = []
val_history = []

# Clear all buffers
for m in models:
    m.memory.memory.clear()
    m.update_target_net()

s = init_empty_state(HEIGHT, WIDTH, NUM_AGENTS)

with open(LOG_FILE, "a") as f:
    f.write(f"\n=== STARTING DECENTRALIZED MAX PERF (N={NUM_AGENTS}): {MODEL_TYPE} ===\n")
    f.write("Step   | Loss    | Avg_V   | Max_V   | Delta | Act% | AvgR_Learn | AvgR_Near | Status | RAM:{ram_mb:.0f}MB\n")

pbar = tqdm(range(int(START_STEP), int(MAX_STEPS)), disable=True)
for step in pbar:
    
    # --- 1. Action & Logging ---
    active_agent_idx = s.get_random_agent_id()


    # Log Team Value
    with torch.no_grad():
        current_val_pred = 0.0
        for i in range(NUM_AGENTS):
            p_i = s.agent_position(i)
            current_val_pred += models[i].get_value(s, agent_pos=p_i)
    val_history.append(current_val_pred)
    
    if random.random() < EPSILON:
        move_action = random_policy(s, active_agent_idx)
    else:
        move_action = get_best_action_decentralized_lookahead(models, s, active_agent_idx)
        
    # --- 2. TRANSITION A: Move (s -> s_mid) ---
    new_pos = np.clip(s.agent_position(active_agent_idx) + move_action.vector, [0, 0], [HEIGHT-1, WIDTH-1])
    s_mid = s.copy()
    s_mid.set_agent_position(active_agent_idx, new_pos)
    
    # Store Experience: Each Agent stores its OWN perspective in its OWN buffer
    for i in range(NUM_AGENTS):
        p_curr = s.agent_position(i)
        p_next = s_mid.agent_position(i)
        models[i].add_experience(s, s_mid, 0.0, agent_pos=p_curr, agent_pos_next=p_next)
    
    # --- 3. TRANSITION B: Pick/Spawn (s_mid -> s_next) ---
    s_next = s_mid.copy()
    
    picker_reward = 0.0
    other_reward = 0.0
    
    if s_next.apples[tuple(new_pos)] > 0:
        s_next.remove_apple_at(new_pos)
        picker_reward = -1.0
        other_reward = 2.0 / (NUM_AGENTS - 1)
        
    spawn_apples(s_next, SPAWN_PROB)
    despawn_apples(s_next, DESPAWN_PROB)
    
    # Store Transition B
    # 1. Active Agent (Picker)
    p_mid = s_mid.agent_position(active_agent_idx)
    p_next = s_next.agent_position(active_agent_idx)
    models[active_agent_idx].add_experience(s_mid, s_next, picker_reward, agent_pos=p_mid, agent_pos_next=p_next)
    
    # 2. Others
    for i in range(NUM_AGENTS):
        if i != active_agent_idx:
            p_mid_i = s_mid.agent_position(i)
            p_next_i = s_next.agent_position(i)
            models[i].add_experience(s_mid, s_next, other_reward, agent_pos=p_mid_i, agent_pos_next=p_next_i)
            
    # --- 4. Train ALL N Models ---
    total_loss = 0.0
    train_count = 0
    
    for m in models:
        # Train if buffer ready
        if len(m.memory) >= BATCH_SIZE:
            # Gradient Clipping logic would ideally be here if we had access to optimizer
            l = m.train_batch(BATCH_SIZE)
            if l is not None:
                total_loss += l
                train_count += 1
                
    if train_count > 0:
        loss_history.append(total_loss / train_count)
    else:
        total_loss = 0.0
        
    if step % TARGET_UPDATE_FREQ == 0:
        for m in models:
            m.update_target_net()
        
    s = s_next
    
    # --- Save Models ---
    if step % SAVE_FREQ == 0:
        for i, m in enumerate(models):
            model_path = OUT_FOLDER / "models" / f"model_agent{i}_step{step}.pt"
            model_path.parent.mkdir(parents=True, exist_ok=True)
            torch.save(m.state_dict(), model_path)
    # --- 5. Eval ---
    if step % EVAL_FREQ == 0 and step > 0:
        recent_vals = val_history[-EVAL_FREQ:]
        avg_v = np.mean(recent_vals)
        max_v = np.max(recent_vals)
        avg_loss = total_loss / train_count if train_count > 0 else 0
        
        # === DIAGNOSTIC START ===
        deltas = []
        correct_actions = 0
        n_test = 100
        for _ in range(n_test):
            s_test = init_empty_state(HEIGHT, WIDTH, NUM_AGENTS)
            s_test.apples[:] = 0
            pos = s_test.agent_position(0)
            
            s_on = s_test.copy()
            s_on.apples[pos[0], pos[1]] = 1
            deltas.append(models[0].get_value(s_on, agent_pos=pos) - models[0].get_value(s_test, agent_pos=pos))
            
            apple_r, apple_c = np.random.randint(0, HEIGHT), np.random.randint(0, WIDTH)
            if (apple_r, apple_c) != tuple(pos):
                s_test.apples[apple_r, apple_c] = 1
                best_val, best_act = -1e9, None
                for act in MoveAction:
                    npos = np.clip(pos + act.vector, [0,0], [HEIGHT-1, WIDTH-1])
                    st = s_test.copy()
                    st.set_agent_position(0, npos)
                    tv = sum(models[i].get_value(st, agent_pos=st.agent_position(i)) for i in range(NUM_AGENTS))
                    if tv > best_val:
                        best_val, best_act = tv, act
                if best_act is None:
                    best_act = MoveAction.get_random_action()
                new_p = np.clip(pos + best_act.vector, [0,0], [HEIGHT-1, WIDTH-1])
                old_d = abs(pos[0]-apple_r) + abs(pos[1]-apple_c)
                new_d = abs(new_p[0]-apple_r) + abs(new_p[1]-apple_c)
                if new_d < old_d:
                    correct_actions += 1
        delta_mean = np.mean(deltas)
        act_acc = correct_actions
        # === DIAGNOSTIC END ===
        
        r_learned = run_evaluation_episode(learned_policy_wrapper, EVAL_STEPS)
        r_nearest = run_evaluation_episode(nearest_policy, EVAL_STEPS)
        
        status = ""
        if r_learned > r_nearest:
            status = "BEAT_NEAREST"
        ram_mb = get_ram_usage()
        with open(LOG_FILE, "a") as f:
            f.write(f"{step:<6} | {avg_loss:.5f} | {avg_v:6.2f}  | {max_v:6.2f}  | {delta_mean:5.2f} | {act_acc:3d}% | {r_learned:.4f}     | {r_nearest:.4f}    | {status} | RAM:{ram_mb:.0f}MB\n")
            
        pbar.set_description(f"L:{avg_loss:.4f} | V:{avg_v:.1f} | D:{delta_mean:.2f} | R:{r_learned:.2f}")